In [100]:
import pandas as pd
import numpy as np
import joblib
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import re

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import seaborn as sns

### Importação de Dados

In [101]:
df = pd.read_csv('../data/raw/Financial_Transactions_By_User.csv')   

In [102]:
df.head()

,UserID,Date,Transaction Description,Category,Amount,Type
0,U001,2025-01-02,Salário Mensal,Salary,11362.00,Income
1,U001,2025-01-11,Loja de Roupas,Shopping,417.87,Expense
2,U001,2025-01-14,Uber,Travel,45.54,Expense
3,U001,2025-01-13,Show,Entertainment,89.83,Expense
4,U001,2025-01-10,Restaurante,Food & Drink,30.73,Expense


In [103]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21451 entries, 0 to 21450
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UserID                   21451 non-null  str    
 1   Date                     21451 non-null  str    
 2   Transaction Description  21451 non-null  str    
 3   Category                 21451 non-null  str    
 4   Amount                   21451 non-null  float64
 5   Type                     21451 non-null  str    
dtypes: float64(1), str(5)
memory usage: 1005.6 KB


In [104]:
df['Category'].value_counts()

Category
Food & Drink        5793
Shopping            4426
Travel              2875
Utilities           2374
Entertainment       2236
Health & Fitness    1392
Investment           888
Rent                 867
Salary               600
Name: count, dtype: int64

### Renomeação e Mapeamento

In [105]:
df.columns

Index(['UserID', 'Date', 'Transaction Description', 'Category', 'Amount',
       'Type'],
      dtype='str')

In [106]:
categorias_br = {"Food & Drink": "Alimentação",
                 "Utilities": "Utilitários",
                 "Rent": "Aluguel",
                 "Investment": "Investimento",
                 "Shopping": "Compras",
                 "Health & Fitness": "Saúde",
                 "Entertainment": "Entretenimento",
                 "Travel": "Trajeto",
                 "Salary": "Salário",
                 "Other": "Outros"}

In [107]:
df['Category'] = df['Category'].map(categorias_br)

In [108]:
df

,UserID,Date,Transaction Description,Category,Amount,Type
0,U001,2025-01-02,Salário Mensal,Salário,11362.00,Income
1,U001,2025-01-11,Loja de Roupas,Compras,417.87,Expense
2,U001,2025-01-14,Uber,Trajeto,45.54,Expense
3,U001,2025-01-13,Show,Entretenimento,89.83,Expense
4,U001,2025-01-10,Restaurante,Alimentação,30.73,Expense
...,...,...,...,...,...,...
21446,U100,2025-06-08,Supermercado,Alimentação,167.87,Expense
21447,U100,2025-06-06,Loja de Roupas,Compras,225.63,Expense
21448,U100,2025-06-19,Conta de Água,Utilitários,323.15,Expense
21449,U100,2025-06-11,Cafeteria,Alimentação,166.38,Expense


In [109]:
tipo_transacao_br = {
    'Income': 'Receita',
    'Expense': 'Despesa'
}

In [110]:
df['Type'] = df['Type'].map(tipo_transacao_br)

In [111]:
df

,UserID,Date,Transaction Description,Category,Amount,Type
0,U001,2025-01-02,Salário Mensal,Salário,11362.00,Receita
1,U001,2025-01-11,Loja de Roupas,Compras,417.87,Despesa
2,U001,2025-01-14,Uber,Trajeto,45.54,Despesa
3,U001,2025-01-13,Show,Entretenimento,89.83,Despesa
4,U001,2025-01-10,Restaurante,Alimentação,30.73,Despesa
...,...,...,...,...,...,...
21446,U100,2025-06-08,Supermercado,Alimentação,167.87,Despesa
21447,U100,2025-06-06,Loja de Roupas,Compras,225.63,Despesa
21448,U100,2025-06-19,Conta de Água,Utilitários,323.15,Despesa
21449,U100,2025-06-11,Cafeteria,Alimentação,166.38,Despesa


In [112]:
df['Date'] = pd.to_datetime(df['Date'])

In [113]:
df['Date'] = df['Date'].dt.strftime('%d/%m/%Y')

In [114]:
df['Date']

0        02/01/2025
1        11/01/2025
2        14/01/2025
3        13/01/2025
4        10/01/2025
            ...    
21446    08/06/2025
21447    06/06/2025
21448    19/06/2025
21449    11/06/2025
21450    22/06/2025
Name: Date, Length: 21451, dtype: str

### Injeção de termos

In [115]:
palavras_reais = {
    'Alimentação': [
        'ifood', 'restaurante', 'padaria', 
        'mcdonalds', 'outback', 'hortifruti', 'uber eats', 'rappi delivery' # <--- Adicionado aqui!
    ],
    'Utilitários': ['conta luz enel', 'conta agua sabesp', 'internet claro', 'fatura vivo', 'gas comgas'],
    'Aluguel': ['pagamento aluguel', 'quinto andar', 'imobiliaria', 'condominio mensal', 'taxa seguro incendio'],
    'Investimento': ['tesouro direto', 'aporte rico corretora', 'investimento xp', 'compra acao b3', 'cdb nubank'],
    'Compras': ['amazon brasil', 'mercado livre', 'magalu', 'zara compras', 'shopee pagamento', 'aliexpress'],
    'Outros': ['transferencia pix', 'tarifa bancaria', 'saque caixa eletronico', 'diversos', 'reembolso'],
    'Entretenimento': ['netflix assina', 'spotify mensal', 'cinema cinemark', 'steam games', 'ingresso show'],
    'Saúde': ['farmacia droga raia', 'consulta medica', 'exame laboratorio', 'drogalis', 'plano saude'],
    'Salário': ['deposito salario', 'pagamento empresa', 'holerite mensal', 'transferencia proventos'],
    'Trajeto': ['uber', '99app corrida', 'posto shell combustivel', 'estapar estacionamento', 'recarga bilhete unico']
}

In [116]:
def injetar_padrao_real(row):
    cat = row['Category']
    if cat in palavras_reais:
        termo = np.random.choice(palavras_reais[cat])
        return termo
    return row['Transaction Description']

df['Transaction Description'] = df.apply(injetar_padrao_real, axis=1)

### Limpeza de Texto (REGEX)

In [ ]:
def limpar_texto(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = texto.strip()
    return texto

df['Transaction Description'] = df['Transaction Description'].apply(limpar_texto)

In [118]:
df[df['Category'] == "Alimentação"]['Transaction Description']

4            hortifruti
7                 ifood
13              padaria
15              padaria
17            uber eats
              ...      
21441        hortifruti
21443         uber eats
21445         mcdonalds
21446    rappi delivery
21449        hortifruti
Name: Transaction Description, Length: 5793, dtype: str

### Vetorização (TF-IDF) e Classificação

In [119]:
df['Category'].unique()

<StringArray>
[       'Salário',        'Compras',        'Trajeto', 'Entretenimento',
    'Alimentação',        'Aluguel',    'Utilitários',          'Saúde',
   'Investimento']
Length: 9, dtype: str

In [120]:
X_train, X_test, y_train, y_test = train_test_split(df['Transaction Description'], df['Category'], test_size=0.2, random_state=2)

In [121]:
pipeline_nlp2 = Pipeline([
    ('vetorizador', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('classificador', RandomForestClassifier(class_weight='balanced',random_state=42))
])

pipeline_nlp2.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vetorizador', ...), ('classificador', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](9,)","['Alimentação','Aluguel','Compras',...,'Saúde','Trajeto','Utilitários']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [122]:
y_pred = pipeline_nlp2.predict(X_test)

print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test, y_pred))

--- RELATÓRIO DE CLASSIFICAÇÃO ---
                precision    recall  f1-score   support

   Alimentação       1.00      1.00      1.00      1171
       Aluguel       1.00      1.00      1.00       177
       Compras       1.00      1.00      1.00       896
Entretenimento       1.00      1.00      1.00       426
  Investimento       1.00      1.00      1.00       177
       Salário       1.00      1.00      1.00       100
         Saúde       1.00      1.00      1.00       295
       Trajeto       1.00      1.00      1.00       571
   Utilitários       1.00      1.00      1.00       478

      accuracy                           1.00      4291
     macro avg       1.00      1.00      1.00      4291
  weighted avg       1.00      1.00      1.00      4291



In [123]:
def predizer_categoria(texto, pipeline, limite_confianca=0.50):
    texto_limpo = limpar_texto(texto)

    probabilidades_categoria = pipeline.predict_proba([texto_limpo])[0]
    maior_probabilidade = np.max(probabilidades_categoria)

    if maior_probabilidade < limite_confianca:
        return "Outros"
    
    return pipeline.classes_[np.argmax(probabilidades_categoria)]

In [133]:
probs = pipeline_nlp2.predict_proba(["Uber"])[0]
categorias = pipeline_nlp2.classes_

df_probs = pd.DataFrame({
    'Categoria': categorias,
    'Probabilidade (%)': (probs * 100).round(2)
}).sort_values(by='Probabilidade (%)', ascending=False).reset_index(drop=True)

df_probs

,Categoria,Probabilidade (%)
0,Trajeto,100.0
1,Alimentação,0.0
2,Aluguel,0.0
3,Entretenimento,0.0
4,Compras,0.0
5,Investimento,0.0
6,Salário,0.0
7,Saúde,0.0
8,Utilitários,0.0


In [125]:
print(predizer_categoria("Uber", pipeline_nlp2))

Trajeto


In [126]:
pipeline_nlp2.predict_proba(["ifood sushi bar"])[0]


array([1., 0., 0., 0., 0., 0., 0., 0., 0.])

In [127]:
pipeline_nlp2.predict(["ifood sushi bar"])

array(['Alimentação'], dtype=object)

In [128]:
pipeline_nlp2.predict(["PIX 'José Carlos'"])

array(['Alimentação'], dtype=object)

In [129]:
pipeline_nlp2.predict(["DROGASIL SP FARMÁCIA"])

array(['Alimentação'], dtype=object)